<a href="https://colab.research.google.com/github/Swayam17o5/DL1/blob/main/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers.optimization import get_scheduler
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
print("Path to dataset files:", path)
print(os.listdir(path))

train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df  = pd.read_csv(os.path.join(path, "test.csv"))

train_df.rename(columns={"Class Index": "label", "Title": "title", "Description": "description"}, inplace=True)
test_df.rename(columns={"Class Index": "label", "Title": "title", "Description": "description"}, inplace=True)

train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"]  = test_df["title"]  + " " + test_df["description"]

train_df["label"] = train_df["label"] - 1
test_df["label"]  = test_df["label"]  - 1

print("Train size:", len(train_df))
print("Test size: ", len(test_df))
print("\nClass distribution:\n", train_df["label"].value_counts())
train_df.head(3)

TRAIN_SIZE = 4000
TEST_SIZE  = 400

train_df = train_df.groupby("label").sample(n=TRAIN_SIZE, random_state=42).reset_index(drop=True)
test_df  = test_df.groupby("label").sample(n=TEST_SIZE,  random_state=42).reset_index(drop=True)

print("Sampled train size:", len(train_df))
print("Sampled test size: ", len(test_df))

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

MAX_LEN = 128

class NewsDataset(Dataset):
    """Custom PyTorch Dataset for tokenizing and batching news text."""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = NewsDataset(train_df["text"].tolist(), train_df["label"].tolist(), tokenizer, MAX_LEN)
test_dataset  = NewsDataset(test_df["text"].tolist(),  test_df["label"].tolist(),  tokenizer, MAX_LEN)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches: ", len(test_loader))

NUM_CLASSES = 4

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print("Using device:", device)

EPOCHS    = 3
LR        = 2e-5
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    return avg_loss

def evaluate(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc, all_preds, all_labels

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_acc, _, _ = evaluate(model, test_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Val Accuracy: {val_acc:.4f}")

class_names = ["World", "Sports", "Business", "Sci/Tech"]

final_acc, preds, labels = evaluate(model, test_loader, device)

print(f"\nFinal Test Accuracy: {final_acc:.4f}")
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=class_names))

def predict(text, model, tokenizer, device):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred    = torch.argmax(outputs.logits, dim=1).item()

    return class_names[pred]

samples = [
    "NASA launches new Mars rover to explore the surface of the red planet.",
    "Stock markets fell sharply as inflation data came in higher than expected.",
    "UN holds emergency meeting over ongoing conflict in the Middle East.",
    "India won the cricket world cup by 10 wickets"
]

for sample in samples:
    label = predict(sample, model, tokenizer, device)
    print(f"Text   : {sample[:70]}...")
    print(f"Predicted: {label}\n")

100%|██████████| 11.4M/11.4M [00:01<00:00, 6.40MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/amananandrai/ag-news-classification-dataset/versions/2
['test.csv', 'train.csv']
Train size: 120000
Test size:  7600

Class distribution:
 label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64
Sampled train size: 16000
Sampled test size:  1600


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Train Loss: 0.1898 | Val Accuracy: 0.9163

--- Epoch 3/3 ---


Evaluating: 100%|██████████| 50/50 [00:06<00:00,  7.67it/s]


Train Loss: 0.1350 | Val Accuracy: 0.9156


Evaluating: 100%|██████████| 50/50 [00:06<00:00,  7.61it/s]


Final Test Accuracy: 0.9156

Classification Report:
              precision    recall  f1-score   support

       World       0.93      0.93      0.93       400
      Sports       0.98      0.97      0.97       400
    Business       0.89      0.86      0.87       400
    Sci/Tech       0.87      0.91      0.89       400

    accuracy                           0.92      1600
   macro avg       0.92      0.92      0.92      1600
weighted avg       0.92      0.92      0.92      1600

Text   : NASA launches new Mars rover to explore the surface of the red planet....
Predicted: Sci/Tech

Text   : Stock markets fell sharply as inflation data came in higher than expec...
Predicted: Business

Text   : UN holds emergency meeting over ongoing conflict in the Middle East....
Predicted: World

Text   : India won the cricket world cup by 10 wickets...
Predicted: Sports



In [ ]:
Dataset Loading

Used:

kagglehub.dataset_download()
Downloaded AG News dataset

Loaded:

train.csv, test.csv

Renamed columns:

Class Index → label
Title → title
Description → description

Combined text:

text = title + description

👉 Model uses full article text for classification

🔹 Label Processing
Original labels: 1–4

Converted to:

0–3

Done using:

label = label - 1

👉 Required for model compatibility

🔹 Data Sampling

Selected smaller dataset:

4000 samples per class (train)
400 samples per class (test)

Used:

groupby("label").sample()

👉 Makes training faster and balanced

🔹 Tokenization (Text → Input IDs)

Used:

DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
Converts text into:
input_ids (word tokens)
attention_mask (important tokens)

Max length:

MAX_LEN = 128

👉 Ensures fixed-length input for model

🔹 Custom Dataset Class

Created:

class NewsDataset(Dataset)
Defines:
__len__() → dataset size

__getitem__() → returns:

input_ids
attention_mask
label

👉 Helps PyTorch load data in batches

🔹 DataLoader

Used:

DataLoader(...)
Batch size = 32
Shuffle = True (training)

👉 Feeds data batch-by-batch to model

🔹 Model Architecture

Used pre-trained model:

DistilBertForSequenceClassification

Set:

num_labels = 4

👉 Classes:

World
Sports
Business
Sci/Tech
🔹 Device Setup

Checked:

torch.cuda.is_available()
Used:
GPU if available
else CPU

👉 Faster training on GPU

🔹 Optimizer & Scheduler

Optimizer:

AdamW

Learning rate:

2e-5

Scheduler:

get_scheduler("linear")

👉 Gradually reduces learning rate

🔹 Training Loop
Function:
train_epoch()
For each batch:
Move data to device

Forward pass:

outputs = model(...)

Compute loss:

loss = outputs.loss

Backprop:

loss.backward()

Update weights:

optimizer.step()
scheduler.step()

👉 Model learns from data

🔹 Evaluation
Function:
evaluate()

Disable gradients:

torch.no_grad()

Predict:

preds = argmax(outputs.logits)

Compute:

accuracy_score()

👉 Measures model performance

🔹 Training Process

Loop:

for epoch in range(1, EPOCHS+1)
Prints:
Training loss
Validation accuracy

👉 Tracks improvement over epochs

🔹 Final Evaluation

Prints:

Final Test Accuracy
classification_report
Report includes:
Precision
Recall
F1-score
🔹 Prediction Function

Defined:

predict(text)
Steps:
Tokenize input
Pass to model

Get class using:

argmax()

👉 Converts raw text → predicted category

🔹 Sample Predictions

Example inputs:

"NASA launches rover" → Sci/Tech
"Stock market falls" → Business
"India wins match" → Sports

👉 Shows real-world usage